<a href="https://colab.research.google.com/github/santiagolondono-max/Yahtzee-Montecarlo/blob/main/Simulaci%C3%B3n_Montecarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
#  SIMULADOR YAHTZEE - MÉTODO MONTECARLO + CADENA DE MARKOV
#  Asignatura: Simulación
#  Descripción: Simulación del juego Yahtzee clásico para 2 jugadores
#               usando el método de Montecarlo para la toma de decisiones
#               y cadenas de Markov para modelar transiciones de estado.
# =============================================================================

import random
import itertools
from collections import Counter

# ─────────────────────────────────────────────────────────────────────────────
# MÓDULO 1: CLASE DADO
# Representa un dado estándar de 6 caras con distribución uniforme U(1,6)
# ─────────────────────────────────────────────────────────────────────────────
class Dado:
    """
    Simula un dado estándar de 6 caras.
    La distribución de probabilidad es uniforme: P(X=k) = 1/6 para k ∈ {1,2,3,4,5,6}
    Esto representa el núcleo del método Montecarlo: generación de números aleatorios.
    """
    def __init__(self):
        self.valor = 0

    def lanzar(self):
        """Lanza el dado. Retorna un valor aleatorio uniforme entre 1 y 6."""
        self.valor = random.randint(1, 6)
        return self.valor

    def __repr__(self):
        return f"[{self.valor}]"


# ─────────────────────────────────────────────────────────────────────────────
# MÓDULO 2: CLASE MARCADOR
# Gestiona las 13 categorías de puntuación del Yahtzee clásico
# ─────────────────────────────────────────────────────────────────────────────
class Marcador:
    """
    Marcador clásico de Yahtzee con 13 categorías.
    Sección superior: Ases, Doses, Treses, Cuatros, Cincos, Seises (+ Bono si >= 63)
    Sección inferior: Trío, Cuádruple, Full House, Esc. Corta, Esc. Larga, Yahtzee, Azar
    """

    CATEGORIAS = [
        "ases", "doses", "treses", "cuatros", "cincos", "seises",
        "trio", "cuadruple", "full_house", "escalera_corta",
        "escalera_larga", "yahtzee", "azar"
    ]

    NOMBRES = {
        "ases": "Ases", "doses": "Doses", "treses": "Treses",
        "cuatros": "Cuatros", "cincos": "Cincos", "seises": "Seises",
        "trio": "Trío", "cuadruple": "Cuádruple",
        "full_house": "Full House", "escalera_corta": "Escalera Corta",
        "escalera_larga": "Escalera Larga", "yahtzee": "Yahtzee", "azar": "Azar"
    }

    def __init__(self):
        # None = categoría libre, número = puntaje anotado
        self.puntajes = {cat: None for cat in self.CATEGORIAS}

    def calcular_puntaje(self, categoria, dados):
        """
        Calcula el puntaje que obtendría 'categoria' con la combinación 'dados'.
        No modifica el marcador — solo calcula.
        """
        conteo = Counter(dados)
        valores = sorted(dados)

        if categoria == "ases":
            return conteo.get(1, 0) * 1
        elif categoria == "doses":
            return conteo.get(2, 0) * 2
        elif categoria == "treses":
            return conteo.get(3, 0) * 3
        elif categoria == "cuatros":
            return conteo.get(4, 0) * 4
        elif categoria == "cincos":
            return conteo.get(5, 0) * 5
        elif categoria == "seises":
            return conteo.get(6, 0) * 6
        elif categoria == "trio":
            if max(conteo.values()) >= 3:
                return sum(dados)
            return 0
        elif categoria == "cuadruple":
            if max(conteo.values()) >= 4:
                return sum(dados)
            return 0
        elif categoria == "full_house":
            vals = sorted(conteo.values())
            if vals == [2, 3]:
                return 25
            return 0
        elif categoria == "escalera_corta":
            unicos = sorted(set(dados))
            for inicio in range(1, 4):
                seq = list(range(inicio, inicio + 4))
                if all(v in unicos for v in seq):
                    return 30
            return 0
        elif categoria == "escalera_larga":
            unicos = sorted(set(dados))
            if unicos == [1,2,3,4,5] or unicos == [2,3,4,5,6]:
                return 40
            return 0
        elif categoria == "yahtzee":
            if len(conteo) == 1:
                return 50
            return 0
        elif categoria == "azar":
            return sum(dados)
        return 0

    def anotar(self, categoria, dados):
        """Anota el puntaje en la categoría indicada."""
        if self.puntajes[categoria] is not None:
            raise ValueError(f"La categoría '{categoria}' ya fue usada.")
        self.puntajes[categoria] = self.calcular_puntaje(categoria, dados)

    def categorias_libres(self):
        """Retorna la lista de categorías aún disponibles."""
        return [c for c in self.CATEGORIAS if self.puntajes[c] is None]

    def puntaje_total(self):
        """Suma todos los puntajes anotados, incluyendo bono superior si aplica."""
        total = 0
        suma_superior = 0
        cats_sup = ["ases", "doses", "treses", "cuatros", "cincos", "seises"]
        for cat in cats_sup:
            if self.puntajes[cat] is not None:
                suma_superior += self.puntajes[cat]
        bono = 35 if suma_superior >= 63 else 0
        for cat in self.CATEGORIAS:
            if self.puntajes[cat] is not None:
                total += self.puntajes[cat]
        return total + bono

    def mostrar(self, nombre_jugador):
        """Imprime el marcador formateado en consola."""
        print(f"\n{'═'*40}")
        print(f"  MARCADOR — {nombre_jugador}")
        print(f"{'═'*40}")
        print(f"  {'SECCIÓN SUPERIOR':25s} PUNTOS")
        print(f"  {'─'*35}")
        cats_sup = ["ases","doses","treses","cuatros","cincos","seises"]
        suma_sup = 0
        for cat in cats_sup:
            pts = self.puntajes[cat]
            if pts is not None:
                suma_sup += pts
            print(f"  {self.NOMBRES[cat]:25s} {str(pts) if pts is not None else '---':>6}")
        bono = 35 if suma_sup >= 63 else 0
        print(f"  {'Subtotal superior':25s} {suma_sup:>6}")
        print(f"  {'Bono (+35 si >= 63)':25s} {bono:>6}")
        print(f"  {'─'*35}")
        print(f"  {'SECCIÓN INFERIOR':25s} PUNTOS")
        print(f"  {'─'*35}")
        cats_inf = ["trio","cuadruple","full_house","escalera_corta","escalera_larga","yahtzee","azar"]
        for cat in cats_inf:
            pts = self.puntajes[cat]
            print(f"  {self.NOMBRES[cat]:25s} {str(pts) if pts is not None else '---':>6}")
        print(f"  {'─'*35}")
        print(f"  {'TOTAL':25s} {self.puntaje_total():>6}")
        print(f"{'═'*40}")


# ─────────────────────────────────────────────────────────────────────────────
# MÓDULO 3: CADENA DE MARKOV
# Modela las transiciones de estado entre combinaciones de dados
# Estado = tupla de valores de dados guardados
# Transición = relanzamiento de dados no guardados
# ─────────────────────────────────────────────────────────────────────────────
class CadenaMarkov:
    """
    Modela el juego como una cadena de Markov discreta.
    - Estado: combinación actual de dados (tupla ordenada)
    - Transición: relanzamiento → nuevo estado con cierta probabilidad

    La propiedad de Markov se cumple porque el estado futuro (próximo lanzamiento)
    solo depende del estado actual (dados guardados), no del historial previo.
    """

    def __init__(self, n_simulaciones=5000):
        self.n_simulaciones = n_simulaciones
        self.historial_transiciones = []  # Registro de transiciones observadas

    def probabilidad_transicion(self, dados_guardados, n_relanzar):
        """
        Estima la distribución de probabilidad del siguiente estado
        dado el estado actual (dados_guardados) y el número de dados a relanzar.
        Retorna un diccionario {estado_destino: probabilidad}.
        """
        conteo_estados = Counter()
        for _ in range(self.n_simulaciones):
            nuevos = tuple(sorted(
                list(dados_guardados) + [random.randint(1,6) for _ in range(n_relanzar)]
            ))
            conteo_estados[nuevos] += 1
        total = sum(conteo_estados.values())
        return {estado: cnt/total for estado, cnt in conteo_estados.items()}

    def registrar_transicion(self, estado_origen, estado_destino):
        """Registra una transición observada para análisis estadístico."""
        self.historial_transiciones.append((estado_origen, estado_destino))

    def estadisticas_transiciones(self):
        """Retorna un resumen de las transiciones más frecuentes observadas."""
        if not self.historial_transiciones:
            return {}
        conteo = Counter(self.historial_transiciones)
        total = len(self.historial_transiciones)
        return {trans: cnt/total for trans, cnt in conteo.most_common(10)}


# ─────────────────────────────────────────────────────────────────────────────
# MÓDULO 4: MOTOR MONTECARLO
# Núcleo de la inteligencia del simulador
# Toma decisiones óptimas mediante simulación masiva de escenarios
# ─────────────────────────────────────────────────────────────────────────────
class MotorMontecarlo:
    """
    Motor de decisiones basado en el método de Montecarlo.

    Principio: ante una decisión, simula N escenarios aleatorios para cada opción
    y elige la que maximiza el valor esperado de puntuación.

    Decisiones que resuelve:
    1. ¿Qué dados guardar entre lanzamientos? (simular_retencion)
    2. ¿En qué categoría anotar al final del turno? (elegir_categoria)
    """

    def __init__(self, n_simulaciones=8000):
        self.n_simulaciones = n_simulaciones
        self.marcador_ref = None  # Referencia al marcador actual para contexto

    def valor_esperado_retencion(self, dados_guardados, n_relanzar, categorias_libres, marcador):
        """
        Estima el valor esperado de mantener 'dados_guardados' y relanzar
        'n_relanzar' dados, evaluando el mejor puntaje posible en las
        categorías disponibles.
        """
        total_valor = 0
        for _ in range(self.n_simulaciones):
            # Simular relanzamiento
            nuevos = list(dados_guardados) + [random.randint(1,6) for _ in range(n_relanzar)]
            # Calcular el mejor puntaje posible con esta combinación
            mejor = max(
                marcador.calcular_puntaje(cat, nuevos)
                for cat in categorias_libres
            )
            total_valor += mejor
        return total_valor / self.n_simulaciones

    def simular_retencion(self, dados_actuales, marcador):
        """
        Decide qué dados guardar usando Montecarlo.
        Evalúa todas las posibles combinaciones de dados a retener
        y selecciona la que maximiza el valor esperado.

        Retorna: lista de índices de los dados a GUARDAR.
        """
        categorias = marcador.categorias_libres()
        if not categorias:
            return list(range(len(dados_actuales)))

        mejor_valor = -1
        mejor_indices = list(range(len(dados_actuales)))  # Por defecto guardar todos

        n = len(dados_actuales)
        # Evaluar cada subconjunto posible de dados a guardar
        for r in range(n + 1):
            for indices in itertools.combinations(range(n), r):
                dados_guardados = [dados_actuales[i] for i in indices]
                n_relanzar = n - len(indices)

                if n_relanzar == 0:
                    # No relanzar nada: calcular puntaje directo
                    valor = max(
                        marcador.calcular_puntaje(cat, dados_actuales)
                        for cat in categorias
                    )
                else:
                    # Reducir simulaciones para subconjuntos con pocos dados guardados
                    sims = max(500, self.n_simulaciones // (2 ** n_relanzar))
                    total = 0
                    for _ in range(sims):
                        nuevos = dados_guardados + [random.randint(1,6) for _ in range(n_relanzar)]
                        mejor = max(marcador.calcular_puntaje(cat, nuevos) for cat in categorias)
                        total += mejor
                    valor = total / sims

                if valor > mejor_valor:
                    mejor_valor = valor
                    mejor_indices = list(indices)

        return mejor_indices

    def elegir_categoria(self, dados, marcador):
        """
        Elige la categoría donde anotar al final del turno.
        Usa Montecarlo para estimar el costo de oportunidad de cada categoría:
        anotar en una categoría ahora vs. reservarla para más adelante.

        Retorna: nombre de la categoría elegida.
        """
        categorias = marcador.categorias_libres()
        if not categorias:
            return None

        mejor_categoria = None
        mejor_valor = -1

        for cat in categorias:
            puntaje_inmediato = marcador.calcular_puntaje(cat, dados)

            # Estimar valor futuro de las categorías restantes si se usa 'cat' ahora
            cats_restantes = [c for c in categorias if c != cat]

            if not cats_restantes:
                valor_total = puntaje_inmediato
            else:
                # Simular partidas futuras para estimar valor de las categorías restantes
                valor_futuro_estimado = 0
                n_sims_cat = 1000
                for _ in range(n_sims_cat):
                    for _ in range(len(cats_restantes)):
                        dados_sim = [random.randint(1,6) for _ in range(5)]
                        valor_futuro_estimado += max(
                            marcador.calcular_puntaje(c, dados_sim)
                            for c in cats_restantes
                        )
                valor_futuro_estimado /= n_sims_cat
                valor_total = puntaje_inmediato + valor_futuro_estimado * 0.3

            if valor_total > mejor_valor:
                mejor_valor = valor_total
                mejor_categoria = cat

        return mejor_categoria


# ─────────────────────────────────────────────────────────────────────────────
# MÓDULO 5: CLASE JUGADOR
# ─────────────────────────────────────────────────────────────────────────────
class Jugador:
    """Representa a un jugador con su nombre, marcador y estadísticas de turno."""

    def __init__(self, nombre):
        self.nombre = nombre
        self.marcador = Marcador()
        self.historial_turnos = []  # [(lanzamientos, categoria_elegida, puntaje)]

    def registrar_turno(self, lanzamientos, categoria, puntaje):
        self.historial_turnos.append({
            "lanzamientos": lanzamientos,
            "categoria": categoria,
            "puntaje": puntaje
        })

    def __repr__(self):
        return f"Jugador({self.nombre}, Total={self.marcador.puntaje_total()})"


# ─────────────────────────────────────────────────────────────────────────────
# MÓDULO 6: CLASE JUEGO YAHTZEE
# Orquesta la partida completa: 13 turnos × 2 jugadores
# ─────────────────────────────────────────────────────────────────────────────
class JuegoYahtzee:
    """
    Orquestador principal del simulador.
    Gestiona la partida completa para 2 jugadores usando el motor Montecarlo
    y la cadena de Markov para todas las decisiones.
    """

    N_TURNOS = 13
    N_DADOS = 5
    MAX_LANZAMIENTOS = 3

    def __init__(self, nombre_j1="Jugador 1", nombre_j2="Jugador 2", verbose=True):
        self.jugadores = [Jugador(nombre_j1), Jugador(nombre_j2)]
        self.motor = MotorMontecarlo(n_simulaciones=6000)
        self.markov = CadenaMarkov(n_simulaciones=3000)
        self.dados = [Dado() for _ in range(self.N_DADOS)]
        self.verbose = verbose
        self.estadisticas = {
            "yahtzees": [0, 0],
            "escaleras_largas": [0, 0],
            "full_houses": [0, 0],
            "puntajes_por_turno": [[], []]
        }

    def lanzar_dados(self, indices=None):
        """
        Lanza los dados indicados por 'indices'.
        Si indices es None, lanza todos.
        Retorna la lista completa de valores actuales.
        """
        if indices is None:
            indices = list(range(self.N_DADOS))
        for i in indices:
            self.dados[i].lanzar()
        return [d.valor for d in self.dados]

    def jugar_turno(self, jugador, num_turno, idx_jugador):
        """
        Ejecuta un turno completo para un jugador.
        Usa Montecarlo en cada decisión: qué guardar y dónde anotar.
        """
        if self.verbose:
            print(f"\n{'─'*50}")
            print(f"  TURNO {num_turno}/13 — {jugador.nombre.upper()}")
            print(f"{'─'*50}")

        historial_lanzamientos = []
        indices_a_lanzar = list(range(self.N_DADOS))  # Al inicio, lanzar todos
        estado_anterior = None

        for num_lanz in range(1, self.MAX_LANZAMIENTOS + 1):
            # ── Lanzamiento ──────────────────────────────────────────────────
            valores = self.lanzar_dados(indices_a_lanzar)
            historial_lanzamientos.append(list(valores))
            estado_actual = tuple(sorted(valores))

            if self.verbose:
                print(f"\n  Lanzamiento {num_lanz}: {valores}")

            # Registrar transición Markov
            if estado_anterior is not None:
                self.markov.registrar_transicion(estado_anterior, estado_actual)
            estado_anterior = estado_actual

            # ── Último lanzamiento: no hay decisión de retención ─────────────
            if num_lanz == self.MAX_LANZAMIENTOS:
                break

            # ── Montecarlo: decidir qué dados guardar ─────────────────────────
            indices_guardar = self.motor.simular_retencion(valores, jugador.marcador)

            if self.verbose:
                dados_guardados = [valores[i] for i in indices_guardar]
                print(f"  → Montecarlo decide guardar: {dados_guardados}")

            # Los dados a lanzar son los NO guardados
            indices_a_lanzar = [i for i in range(self.N_DADOS) if i not in indices_guardar]

            if not indices_a_lanzar:
                if self.verbose:
                    print(f"  → Se guardan todos los dados. Turno terminado anticipadamente.")
                break

        # ── Montecarlo: elegir categoría ──────────────────────────────────────
        valores_finales = [d.valor for d in self.dados]
        categoria = self.motor.elegir_categoria(valores_finales, jugador.marcador)
        puntaje = jugador.marcador.calcular_puntaje(categoria, valores_finales)
        jugador.marcador.anotar(categoria, valores_finales)

        # Registrar estadísticas
        self.estadisticas["puntajes_por_turno"][idx_jugador].append(puntaje)
        if categoria == "yahtzee" and puntaje == 50:
            self.estadisticas["yahtzees"][idx_jugador] += 1
        if categoria == "escalera_larga" and puntaje == 40:
            self.estadisticas["escaleras_largas"][idx_jugador] += 1
        if categoria == "full_house" and puntaje == 25:
            self.estadisticas["full_houses"][idx_jugador] += 1

        if self.verbose:
            print(f"\n  Dados finales : {valores_finales}")
            nombre_cat = Marcador.NOMBRES[categoria]
            print(f"  → Anota en    : {nombre_cat} → {puntaje} puntos")
            print(f"  Total actual  : {jugador.marcador.puntaje_total()} pts")

        jugador.registrar_turno(historial_lanzamientos, categoria, puntaje)

    def jugar_partida(self):
        """
        Ejecuta la partida completa: 13 turnos por jugador, alternando.
        """
        print("\n" + "═"*50)
        print("   SIMULADOR YAHTZEE — MÉTODO MONTECARLO")
        print("   Distribución uniforme U(1,6) · 2 Jugadores")
        print("═"*50)
        print(f"  {self.jugadores[0].nombre}  vs  {self.jugadores[1].nombre}")
        print("═"*50)

        for turno in range(1, self.N_TURNOS + 1):
            for idx, jugador in enumerate(self.jugadores):
                self.jugar_turno(jugador, turno, idx)

        self.mostrar_resultados()

    def mostrar_resultados(self):
        """Muestra marcadores finales, ganador y estadísticas de la simulación."""
        print("\n\n" + "█"*50)
        print("   RESULTADOS FINALES")
        print("█"*50)

        for jugador in self.jugadores:
            jugador.marcador.mostrar(jugador.nombre)

        pts = [j.marcador.puntaje_total() for j in self.jugadores]
        print("\n" + "═"*50)
        if pts[0] > pts[1]:
            ganador = self.jugadores[0]
            diferencia = pts[0] - pts[1]
        elif pts[1] > pts[0]:
            ganador = self.jugadores[1]
            diferencia = pts[1] - pts[0]
        else:
            ganador = None
            diferencia = 0

        if ganador:
            print(f"\n  🏆  GANADOR: {ganador.nombre.upper()}")
            print(f"      Diferencia: {diferencia} puntos")
        else:
            print(f"\n  🤝  EMPATE — Ambos jugadores: {pts[0]} puntos")

        # ── Estadísticas Montecarlo ───────────────────────────────────────────
        print("\n" + "─"*50)
        print("  ESTADÍSTICAS DE LA SIMULACIÓN")
        print("─"*50)
        for idx, jugador in enumerate(self.jugadores):
            turnos = self.estadisticas["puntajes_por_turno"][idx]
            prom = sum(turnos) / len(turnos) if turnos else 0
            print(f"\n  {jugador.nombre}:")
            print(f"    Puntaje total       : {pts[idx]} pts")
            print(f"    Promedio por turno  : {prom:.1f} pts")
            print(f"    Mejor turno         : {max(turnos)} pts")
            print(f"    Peor turno          : {min(turnos)} pts")
            print(f"    Yahtzees logrados   : {self.estadisticas['yahtzees'][idx]}")
            print(f"    Escaleras largas    : {self.estadisticas['escaleras_largas'][idx]}")
            print(f"    Full Houses         : {self.estadisticas['full_houses'][idx]}")

        # ── Estadísticas Cadena de Markov ─────────────────────────────────────
        print("\n" + "─"*50)
        print("  ANÁLISIS CADENA DE MARKOV")
        print(f"  Total de transiciones de estado registradas: {len(self.markov.historial_transiciones)}")
        trans = self.markov.estadisticas_transiciones()
        if trans:
            print("  Transiciones más frecuentes (top 5):")
            for i, (t, p) in enumerate(list(trans.items())[:5]):
                print(f"    {i+1}. {t[0]} → {t[1]}  (frec. {p:.3f})")
        print("═"*50)


# ─────────────────────────────────────────────────────────────────────────────
# PUNTO DE ENTRADA
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    random.seed(42)  # Semilla fija para reproducibilidad académica
    juego = JuegoYahtzee(
        nombre_j1="Jugador_1",
        nombre_j2="Jugador_2",
        verbose=True
    )
    juego.jugar_partida()
